In [1]:
import os
import gc
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
# files = {
#     10: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_10percent_missing.csv",
#     20: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_20percent_missing.csv",
#     40: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_40percent_missing.csv",
#     80: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_80percent_missing.csv",
#      0: "/kaggle/input/datasets/chetannarware/pemsbay/pemsbaay/pemsbay_0percent_missing.csv"
# }
# print(files[10])
files = {
    10: "/kaggle/input/datasets/chetannarware/loopsea/loop_sea_main/LOOPSEA_10percent_missing.csv",
    20: "/kaggle/input/datasets/chetannarware/loopsea/loop_sea_main/LOOPSEA_20percent_missing.csv",
    40: "/kaggle/input/datasets/chetannarware/loopsea/loop_sea_main/LOOPSEA_40percent_missing.csv",
    80: "/kaggle/input/datasets/chetannarware/loopsea/loop_sea_main/LOOPSEA_80percent_missing.csv",
     0: "/kaggle/input/datasets/chetannarware/loopsea/loop_sea_main/LOOPSEA.csv"
}
print(files[10])

/kaggle/input/datasets/chetannarware/loopsea/loop_sea_main/LOOPSEA_10percent_missing.csv


In [3]:
def create_sequences(data, seq_len=10):
    X, Y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        Y.append(data[i+seq_len])
    return np.array(X), np.array(Y)

In [4]:
def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

In [5]:
class BaseLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.1
        )
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.output_layer(out[:, -1, :])


class BaseRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.1
        )
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.output_layer(out[:, -1, :])


class BaseConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv1d(
            input_dim + hidden_dim,
            4 * hidden_dim,
            kernel_size=3,
            padding=1
        )
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        i, f, o, g = torch.chunk(self.conv(combined), 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        h = self.dropout(h)
        return h, c


class BaseConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.convlstm1 = BaseConvLSTMCell(input_dim=1, hidden_dim=hidden_dim)
        self.convlstm2 = BaseConvLSTMCell(input_dim=hidden_dim, hidden_dim=hidden_dim)
        self.output_head = nn.Conv1d(hidden_dim, 1, kernel_size=1)

    def forward(self, x):
        B, T, F = x.shape
        h1 = torch.zeros(B, self.hidden_dim, F, device=x.device)
        c1 = torch.zeros(B, self.hidden_dim, F, device=x.device)
        h2 = torch.zeros(B, self.hidden_dim, F, device=x.device)
        c2 = torch.zeros(B, self.hidden_dim, F, device=x.device)
        for t in range(T):
            xt = x[:, t, :].unsqueeze(1)
            h1, c1 = self.convlstm1(xt, h1, c1)
            h2, c2 = self.convlstm2(h1, h2, c2)
        return self.output_head(h2).squeeze(1)


class BaseLSTM_Single(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.output_layer(out[:, -1, :])

In [6]:
def compute_loss_baseline(pred, target):
    mae = torch.mean(torch.abs(pred - target))
    mse = torch.mean((pred - target) ** 2)
    return mae + 0.1 * mse


def train_model_baseline(model, train_loader, val_loader, model_type, percent,
                         max_epochs=70, patience=15):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5
    )
    best_loss = float("inf")
    wait = 0
    best_path = f"/kaggle/working/best_base_{model_type}_{percent}.pt"

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}", leave=False)
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = compute_loss_baseline(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                val_loss += compute_loss_baseline(pred, y).item()
        val_loss /= len(val_loader)

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        lr_msg = f"  → LR dropped to {new_lr:.2e}" if new_lr < current_lr else ""
        print(f"Epoch {epoch+1:02d} | Train: {train_loss:.4f} | "
              f"Val: {val_loss:.4f} | LR: {current_lr:.2e}{lr_msg}")

        if val_loss < best_loss:
            best_loss = val_loss
            wait = 0
            torch.save({"model": model.state_dict()}, best_path)
            print("  ✓ Saved best model")
        else:
            wait += 1
            if wait >= patience:
                print("  Early stopping triggered")
                break

    state = torch.load(best_path)["model"]
    model.load_state_dict(state)
    return model



In [7]:
def evaluate_model(model, loader, mean, std):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            pred = model(x)
            preds.append(pred.cpu().numpy())
            trues.append(y.numpy())

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)

    preds_real = preds * std + mean
    trues_real = trues * std + mean

    mae  = mean_absolute_error(trues_real.ravel(), preds_real.ravel())
    rmse = np.sqrt(mean_squared_error(trues_real.ravel(), preds_real.ravel()))
    r2   = r2_score(trues_real.ravel(), preds_real.ravel())
    valid = np.abs(trues_real) > 1e-3
    mape = np.mean(np.abs((trues_real[valid] - preds_real[valid]) /
                           trues_real[valid])) * 100

    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"MAPE : {mape:.2f}%")
    return mae, rmse, r2, mape

In [8]:
def train_single_baseline(percent, model_type="lstm", model_name=None):
    if model_name is None:
        model_name = f"base_{model_type}"

    print(f"\n{'='*30}")
    print(f"BASELINE: {model_type.upper()} | {percent}% MISSING")
    print(f"{'='*30}")

    raw = pd.read_csv(files[percent])
    raw = raw.apply(pd.to_numeric, errors='coerce')

    data = raw.values.astype(np.float32)

    # Warm-start first 200 rows
    col_medians = np.nanmedian(data, axis=0)
    col_medians = np.where(np.isnan(col_medians), 0.0, col_medians)
    for col in range(data.shape[1]):
        nan_rows = np.isnan(data[:200, col])
        if nan_rows.any():
            data[:200, col][nan_rows] = col_medians[col]

    split = int(0.8 * len(data))
    train_data, val_data = data[:split], data[split:]

    # Normalize on observed values only
    mean = np.nanmean(train_data, axis=0, keepdims=True)
    std  = np.nanstd(train_data,  axis=0, keepdims=True)
    mean = np.where(np.isnan(mean), 0.0, mean)
    std  = np.where(np.isnan(std) | (std < 1e-8), 1.0, std)

    # Fill missing with 0
    train_norm = np.where(np.isnan(train_data), 0.0, (train_data - mean) / std)
    val_norm   = np.where(np.isnan(val_data),   0.0, (val_data   - mean) / std)

    SEQ_LEN = 10
    X_tr, Y_tr = create_sequences(train_norm, SEQ_LEN)
    X_vl, Y_vl = create_sequences(val_norm,   SEQ_LEN)

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_tr, dtype=torch.float32),
            torch.tensor(Y_tr, dtype=torch.float32)
        ),
        batch_size=64, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_vl, dtype=torch.float32),
            torch.tensor(Y_vl, dtype=torch.float32)
        ),
        batch_size=64, shuffle=False, num_workers=0
    )

    input_dim = X_tr.shape[2]

    if model_type == "lstm":
        model = BaseLSTM(input_dim=input_dim, hidden_dim=64).to(device)
    elif model_type == "lstm1":
        model = BaseLSTM_Single(input_dim=input_dim, hidden_dim=64).to(device)
    elif model_type == "rnn":
        model = BaseRNN(input_dim=input_dim, hidden_dim=64).to(device)
    elif model_type == "convlstm":
        model = BaseConvLSTM(input_dim=input_dim, hidden_dim=64).to(device)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    print(f"Model: {model_type.upper()}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    model = train_model_baseline(model, train_loader, val_loader, model_type, percent)

    save_path = f"/kaggle/working/{model_name}_{percent}.pt"
    torch.save({"model": model.state_dict(), "mean": mean, "std": std}, save_path)
    print("Saved:", save_path)

    mae, rmse, r2, mape = evaluate_model(model, val_loader, mean, std)
    return mae, rmse, r2, mape

In [9]:
dataset_name = "loopsea"

In [10]:
results = {}

RUN_PERCENT = 20

print(f"\n{'='*10} {RUN_PERCENT}% Missing {'='*10}")

results[RUN_PERCENT] = train_single_baseline(
    percent=RUN_PERCENT,
    model_type="convlstm",   # <-- IMPORTANT: this selects BaseConvLSTM
    model_name="base_convlstm"
)

mae, rmse, r2, mape = results[RUN_PERCENT]

print(f"\nFINAL RESULT {RUN_PERCENT}%")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")


========== 20% Missing ==========

BASELINE: CONVLSTM | 20% MISSING


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


Model: CONVLSTM
Parameters: 148,801


Epoch 1/70:   0%|          | 0/1314 [00:00<?, ?it/s]

Epoch 01 | Train: 0.3806 | Val: 0.3759 | LR: 1.00e-03
  ✓ Saved best model


...

Epoch 68 | Train: 0.3509 | Val: 0.3568 | LR: 1.25e-04


Epoch 69/70:   0%|          | 0/1314 [00:00<?, ?it/s]

Epoch 69 | Train: 0.3509 | Val: 0.3568 | LR: 1.25e-04  → LR dropped to 6.25e-05


Epoch 70/70:   0%|          | 0/1314 [00:00<?, ?it/s]

Epoch 70 | Train: 0.3508 | Val: 0.3567 | LR: 6.25e-05
Saved: /kaggle/working/base_convlstm_20.pt
MAE  : 3.2829
RMSE : 5.8654
R2   : 0.7562
MAPE : 7.43%

FINAL RESULT 20%
MAE  : 3.2829
RMSE : 5.8654
R2   : 0.7562
MAPE : 7.43%
